In [ ]:
# Built-in modules
import math

# Imports from Qiskit
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, MCMTGate, ZGate
from qiskit.visualization import plot_distribution
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Imports from qiskit-ibm-runtime
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler


def grover_oracle(marked_states):
    """Build a Grover oracle for multiple marked states

    Here we assume all input marked states have the same number of bits

    Parameters:
        marked_states (str or list): Marked states of oracle

    Returns:
        QuantumCircuit: Quantum circuit representing Grover oracle
    """
    if not isinstance(marked_states, list):
        marked_states = [marked_states]
    # Compute the number of qubits in circuit
    num_qubits = len(marked_states[0])

    qc = QuantumCircuit(num_qubits)
    # Mark each target state in the input list
    for target in marked_states:
        # Flip target bit-string to match Qiskit bit-ordering
        rev_target = target[::-1]
        # Find the indices of all the '0' elements in bit-string
        zero_inds = [
            ind
            for ind in range(num_qubits)
            if rev_target.startswith("0", ind)
        ]
        # Add a multi-controlled Z-gate with pre- and post-applied X-gates (open-controls)
        # where the target bit-string has a '0' entry
        if zero_inds:
            qc.x(zero_inds)
        qc.compose(MCMTGate(ZGate(), num_qubits - 1, 1), inplace=True)
        if zero_inds:
            qc.x(zero_inds)
    return qc

In [ ]:
marked_states = ["1101001"]

oracle = grover_oracle(marked_states)

grover_op = grover_operator(oracle)

oracle.draw(output="mpl", style="iqp")

In [ ]:
# Theoretical optimal number of Grover iterations
N = 2 ** grover_op.num_qubits
M = len(marked_states)

optimal_num_iterations = math.floor(
    math.pi / (4 * math.asin(math.sqrt(M / N)))
)

print("Theoretical optimal iterations:", optimal_num_iterations)

In [ ]:
def build_grover_circuit(iterations):
    qc = QuantumCircuit(grover_op.num_qubits)

    # Initial superposition
    qc.h(range(grover_op.num_qubits))

    # Grover iterations
    for _ in range(iterations):
        qc.compose(grover_op, inplace=True)

    qc.measure_all()

    return qc

In [ ]:
service = QiskitRuntimeService()

backend = service.backend(name="ibm_fez")

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

noisy_simulator = AerSimulator.from_backend(backend)


iteration_data = {}

for iterations in range(1, optimal_num_iterations + 1):

    # Build logical Grover circuit
    qc = build_grover_circuit(iterations)

    # Transpile for the actual IBM hardware
    optimized = transpile(
        qc,
        backend=noisy_simulator,
        optimization_level=3,
        seed_transpiler=42
    )

    iteration_data[iterations] = {
        "circuit": optimized,
        "depth": optimized.depth(),
        "2q_gates": sum(
            count
            for gate, count in optimized.count_ops().items()
            if gate in ["cx", "ecr", "cz"]
        )
    }

    print(
        f"{iterations:2d} iterations | "
        f"depth = {optimized.depth():3d} | "
        f"2Q gates = {iteration_data[iterations]['2q_gates']}"
    )

In [ ]:
# -----------------------------
# Zero-Noise Extrapolation (ZNE)
# -----------------------------

def fold_circuit(circuit, scale_factor):
    """Globally fold a transpiled circuit to amplify its noise.

    scale_factor = 1: U
    scale_factor = 3: U U^dagger U
    scale_factor = 5: U U^dagger U U^dagger U

    Measurements/barriers are kept outside the folding operation.
    """
    if scale_factor < 1 or scale_factor % 2 == 0:
        raise ValueError("scale_factor must be a positive odd integer")

    if scale_factor == 1:
        return circuit.copy()

    folded = QuantumCircuit(circuit.num_qubits, circuit.num_clbits)

    # Copy operations one-by-one. Measurement/reset/classical operations are
    # not unitary and therefore are not folded.
    for instruction in circuit.data:
        op = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits

        if op.name in {"measure", "reset", "barrier"} or not op.is_unitary():
            folded.append(instruction)
            continue

        # U (U^dagger U)^k
        folded.append(instruction)
        for _ in range((scale_factor - 1) // 2):
            folded.append((op.inverse(), qargs, cargs))
            folded.append((op, qargs, cargs))

    return folded


def distribution_from_counts(counts, shots):
    return {state: count / shots for state, count in counts.items()}


def zne_peak_probability(circuit, shots=100_000, scale_factors=(1, 3, 5)):
    """Run noise-scaled circuits and extrapolate the peak probability to zero noise.

    The state whose probability is largest is determined from the measurement
    data; it is never supplied to the function as a known answer.
    """
    distributions = {}

    for scale in scale_factors:
        folded = fold_circuit(circuit, scale)
        result = noisy_simulator.run(folded, shots=shots).result()
        counts = result.get_counts()
        distributions[scale] = distribution_from_counts(counts, shots)

    # Use the union of observed states across all noise levels.
    states = set().union(*(d.keys() for d in distributions.values()))

    # First identify the state with the highest probability at the least-noisy
    # (scale=1) circuit. This is inferred from the experiment, not hard-coded.
    peak_state = max(states, key=lambda s: distributions[scale_factors[0]].get(s, 0.0))

    x = list(scale_factors)
    y = [distributions[s].get(peak_state, 0.0) for s in scale_factors]

    # Linear Richardson extrapolation: fit p(lambda) = a*lambda + b,
    # then evaluate at lambda = 0 (the ideal zero-noise limit).
    degree = min(1, len(x) - 1)
    coefficients = __import__("numpy").polyfit(x, y, degree)
    zne_probability = __import__("numpy").polyval(coefficients, 0)

    return {
        "peak_state": peak_state,
        "raw_peak_probability": y[0],
        "zne_peak_probability": float(zne_probability),
        "scale_factors": x,
        "scaled_probabilities": y,
        "distributions": distributions
    }


In [ ]:
# Choose which Grover iteration count to inspect.
# This is an experimental setting, NOT an iteration checker.
iterations_to_plot = 1

dist = zne_results[iterations_to_plot]["distributions"][1]

top_20 = dict(sorted(dist.items(), key=lambda item: item[1], reverse=True)[:20])

fig = plot_distribution(top_20, figsize=(14, 5))
ax = fig.axes[0]
ax.tick_params(axis="x", labelrotation=45, labelsize=8)
plt.tight_layout()
display(fig)
plt.close(fig)


In [ ]:
from pathlib import Path

output_dir = Path("figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig = plot_distribution(dist, figsize=(200, 5))
fig.savefig(output_dir / "distribution_zne.png", dpi=300, bbox_inches="tight")
plt.close(fig)
